# 🌐 Phase 2: Common Crawl PPTX Mining

Extracts PPT/PPTX files directly from Common Crawl web archives.

**How it works:**
1. Query CC CDX index for PPT/PPTX files across academic TLDs
2. Get WARC file location (filename, offset, length) for each result
3. Download exact bytes from S3 via HTTP range requests
4. Parse WARC record → extract the actual file
5. Apply filters (2MB, geo, magic bytes)

**Expected yield:** 50K-200K+ unique files across multiple crawls

In [ ]:
# Cell 1: Setup
!pip install -q warcio
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
warnings.filterwarnings('ignore')

DRIVE_DIR = '/content/drive/MyDrive/PPTX_Global_Collection'
SEEN_FILE = os.path.join(DRIVE_DIR, 'seen_tags.txt')
CC_PROGRESS = os.path.join(DRIVE_DIR, 'cc_progress.json')
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Output: {DRIVE_DIR}')

In [ ]:
# Cell 2: Common Crawl config
import gzip, hashlib, io, json, os, re, time, zipfile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

MIN_SIZE = 2 * 1024 * 1024
CC_INDEX_URL = 'https://index.commoncrawl.org'
CC_DATA_URL = 'https://data.commoncrawl.org'

CHINA_KW = ['chinese','china','hong kong','taiwan','beijing','shanghai',
            'mandarin','wuhan','guangzhou','shenzhen','nanjing','zhonghua']

# Academic TLDs across 71 countries — high PPT density
ACADEMIC_TLDS = [
    # Major English-speaking
    '*.edu', '*.ac.uk', '*.edu.au', '*.edu.nz', '*.ac.nz',
    '*.edu.ca', '*.ac.ca',
    # Africa
    '*.ac.za', '*.edu.za', '*.edu.ng', '*.edu.eg', '*.ac.ke',
    '*.edu.ke', '*.ac.ma', '*.edu.gh', '*.edu.tn',
    # Asia
    '*.ac.in', '*.edu.in', '*.ac.jp', '*.ac.kr', '*.edu.sg',
    '*.edu.my', '*.edu.tr', '*.ac.id', '*.edu.ph', '*.edu.pk',
    '*.edu.bd', '*.ac.ae', '*.ac.ir', '*.edu.sa', '*.edu.jo',
    '*.edu.lb', '*.edu.qa', '*.ac.lk', '*.edu.kz',
    '*.ac.th', '*.edu.vn',
    # Europe
    '*.edu.fr', '*.ac.at', '*.edu.it', '*.edu.es', '*.edu.nl',
    '*.edu.ch', '*.ac.be', '*.edu.se', '*.edu.no', '*.edu.dk',
    '*.edu.fi', '*.edu.ie', '*.edu.pt', '*.edu.gr', '*.edu.pl',
    '*.edu.cz', '*.edu.hu', '*.edu.ro', '*.edu.ua', '*.edu.hr',
    '*.edu.rs', '*.edu.ru', '*.ac.ru', '*.edu.bg', '*.edu.sk',
    '*.edu.lt', '*.edu.si', '*.edu.ee',
    # Americas
    '*.edu.mx', '*.edu.br', '*.edu.ar', '*.edu.cl', '*.edu.co',
    '*.edu.pe', '*.edu.ec', '*.edu.ve', '*.edu.cr', '*.edu.uy',
    '*.edu.cu',
    # Generic high-yield
    '*.uni-*.de', '*.univ-*.fr',
]

# PPT MIME types
PPT_MIMES = [
    'application/vnd.ms-powerpoint',
    'application/vnd.openxmlformats-officedocument.presentationml.presentation',
]

session = requests.Session()
retry = Retry(total=3, backoff_factor=2, status_forcelist=[429,500,502,503,504])
session.mount('https://', HTTPAdapter(max_retries=retry))
session.mount('http://', HTTPAdapter(max_retries=retry))

def load_seen():
    tags = set()
    if os.path.exists(SEEN_FILE):
        with open(SEEN_FILE) as f:
            tags = {l.strip() for l in f if l.strip()}
    return tags

def save_tag(tag):
    with open(SEEN_FILE, 'a') as f: f.write(tag + '\n')

def is_china(text):
    if not text: return False
    return any(kw in text.lower() for kw in CHINA_KW)

def has_chinese_chars(fp):
    try:
        with zipfile.ZipFile(fp, 'r') as z:
            for n in z.namelist():
                if 'slide' in n and n.endswith('.xml'):
                    d = z.read(n).decode('utf-8', errors='ignore')
                    if len(re.findall(r'[\u4e00-\u9fff]', d)) > 50: return True
    except: pass
    return False

def valid_ppt(fp):
    try:
        with open(fp, 'rb') as f: h = f.read(8)
        return h[:2] == b'PK' or h[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    except: return False

def load_cc_progress():
    if os.path.exists(CC_PROGRESS):
        with open(CC_PROGRESS) as f: return json.load(f)
    return {'completed_tlds': [], 'crawl_idx': 0, 'downloaded': 0}

def save_cc_progress(p):
    with open(CC_PROGRESS, 'w') as f: json.dump(p, f)

print(f'✅ Config ready: {len(ACADEMIC_TLDS)} TLDs × {len(PPT_MIMES)} MIME types')

In [ ]:
# Cell 3: Get available crawls
print('📋 Fetching available Common Crawl indices...')
r = session.get(f'{CC_INDEX_URL}/collinfo.json', timeout=30)
all_crawls = r.json()
# Use the latest 10 crawls for best coverage
crawls = [c['cdx-api'] for c in all_crawls[:10]]
crawl_names = [c['id'] for c in all_crawls[:10]]
print(f'✅ Using {len(crawls)} most recent crawls:')
for cn in crawl_names:
    print(f'   - {cn}')

In [ ]:
# Cell 4: CDX Query + WARC Download Engine
from warcio.archiveiterator import ArchiveIterator

seen = load_seen()
cc_prog = load_cc_progress()
stats = {'queried': 0, 'found': 0, 'downloaded': 0,
         'small': 0, 'china': 0, 'invalid': 0, 'dup': 0, 'errors': 0}

def query_cdx(cdx_api, tld, mime_type, page=0):
    """Query Common Crawl CDX index for files matching TLD + MIME type."""
    params = {
        'url': tld,
        'matchType': 'domain',
        'filter': f'mime:{mime_type}',
        'output': 'json',
        'page': page,
        'pageSize': 50,
    }
    try:
        r = session.get(cdx_api, params=params, timeout=60)
        if r.ok and r.text.strip():
            lines = r.text.strip().split('\n')
            results = []
            for line in lines:
                try: results.append(json.loads(line))
                except: continue
            return results
    except Exception as e:
        pass
    return []

def download_from_warc(record_info):
    """Download a single file from Common Crawl WARC archive."""
    global seen, stats
    url = record_info.get('url', '')
    filename = record_info.get('filename', '')
    offset = int(record_info.get('offset', 0))
    length = int(record_info.get('length', 0))

    if not all([url, filename, length]): return False

    tag = hashlib.sha1(url.encode()).hexdigest()[:10]
    if tag in seen: stats['dup'] += 1; return False
    if is_china(url): stats['china'] += 1; return False

    # Determine extension from URL
    url_lower = url.lower().split('?')[0]
    ext = '.ppt' if url_lower.endswith('.ppt') else '.pptx'
    safe_url = re.sub(r'[^\w]', '_', url.split('/')[-1][:40])
    out_name = f'{tag}_cc_{safe_url}{ext}'
    dest = os.path.join(DRIVE_DIR, out_name)
    if os.path.exists(dest): seen.add(tag); return False

    try:
        # Byte-range request to get exactly this record from the WARC file
        warc_url = f'{CC_DATA_URL}/{filename}'
        headers = {'Range': f'bytes={offset}-{offset + length - 1}'}
        r = session.get(warc_url, headers=headers, timeout=90)
        if not r.ok: stats['errors'] += 1; return False

        # Parse WARC record to extract the file content
        stream = io.BytesIO(r.content)
        for warc_record in ArchiveIterator(stream):
            if warc_record.rec_type == 'response':
                # Read HTTP response body (the actual file)
                payload = warc_record.content_stream().read()

                # Check size
                if len(payload) < MIN_SIZE: stats['small'] += 1; return False

                # Save to disk
                with open(dest, 'wb') as f: f.write(payload)

                # Validate
                if not valid_ppt(dest):
                    os.remove(dest); stats['invalid'] += 1; return False
                if has_chinese_chars(dest):
                    os.remove(dest); stats['china'] += 1; return False

                seen.add(tag); save_tag(tag)
                stats['downloaded'] += 1
                return True

        stats['errors'] += 1; return False
    except:
        if os.path.exists(dest): os.remove(dest)
        stats['errors'] += 1; return False

print('✅ CDX + WARC engine ready')

In [ ]:
# Cell 5: Run the mining!
TARGET = 200000
completed_tlds = set(cc_prog.get('completed_tlds', []))
start_crawl = cc_prog.get('crawl_idx', 0)

print(f'🚀 Starting Common Crawl mining')
print(f'   Target: {TARGET} files')
print(f'   TLDs: {len(ACADEMIC_TLDS)}')
print(f'   Crawls: {len(crawls)}')
print(f'   Already completed: {len(completed_tlds)} TLDs')
print()

for ci, (cdx_api, crawl_name) in enumerate(zip(crawls[start_crawl:], crawl_names[start_crawl:])):
    if stats['downloaded'] >= TARGET: break
    print(f'\n{"="*60}')
    print(f'🗓️  CRAWL {ci+start_crawl+1}/{len(crawls)}: {crawl_name}')
    print(f'{"="*60}')

    for ti, tld in enumerate(ACADEMIC_TLDS):
        if stats['downloaded'] >= TARGET: break
        tld_key = f'{crawl_name}:{tld}'
        if tld_key in completed_tlds: continue

        print(f'  [{ti+1}/{len(ACADEMIC_TLDS)}] {tld} | Total: {stats["downloaded"]}/{TARGET}')

        for mime in PPT_MIMES:
            if stats['downloaded'] >= TARGET: break
            page = 0

            while stats['downloaded'] < TARGET:
                results = query_cdx(cdx_api, tld, mime, page)
                if not results: break
                stats['queried'] += len(results)
                stats['found'] += len(results)

                for rec in results:
                    if stats['downloaded'] >= TARGET: break
                    download_from_warc(rec)

                if len(results) < 50: break  # Last page
                page += 1
                time.sleep(0.5)  # Be nice to CC servers

            time.sleep(0.3)

        completed_tlds.add(tld_key)
        cc_prog['completed_tlds'] = list(completed_tlds)
        cc_prog['crawl_idx'] = ci + start_crawl
        cc_prog['downloaded'] = stats['downloaded']
        save_cc_progress(cc_prog)

print(f'\n{"="*60}')
print(f'📊 COMMON CRAWL RESULTS')
print(f'{"="*60}')
for k,v in stats.items(): print(f'   {k:20s}: {v}')
print(f'{"="*60}')

with open(os.path.join(DRIVE_DIR, 'cc_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

In [ ]:
# Cell 6: Summary
import glob
files = glob.glob(os.path.join(DRIVE_DIR,'*.pptx')) + glob.glob(os.path.join(DRIVE_DIR,'*.ppt'))
cc_files = [f for f in files if '_cc_' in f]
total_gb = sum(os.path.getsize(f) for f in files) / (1024**3)
print(f'📊 Common Crawl: {len(cc_files)} files')
print(f'📊 Total collection: {len(files)} files | {total_gb:.2f} GB')